# 🎓 ExamGuard AI — YOLOv8 Training

Works **locally on Windows** or on **Google Colab**.
No annotation, no Roboflow — trains directly on your folder structure.

```
ExamCheatingDataset/
├── train/
│   ├── normal act/
│   ├── looking friend/
│   ├── giving object/
│   ├── giving code/
│   └── cheating/
├── valid/   (or val/)
└── test/
```

## 1 — Install YOLOv8

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ultralytics', '-q'])
print('✅ Ultralytics installed')

## 2 — Point to your dataset

**Set `DATASET_ROOT` to the folder that contains `train/`, `valid/`, `test/`.**

- Windows local: something like `r'C:\Users\YourName\Downloads\ExamCheatingDataset'`
- Google Colab: upload your zip first (see note below), then set the extracted path

In [ ]:
import os
from pathlib import Path

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ↓↓↓  SET THIS TO YOUR DATASET FOLDER  ↓↓↓
DATASET_ROOT = r'C:\Users\ROG\Downloads\ExamCheatingDataset'
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ── If running on Google Colab with a zip upload ──────────────────
# Uncomment these lines instead (after uploading zip via the 📁 sidebar):
# import zipfile
# ZIP_PATH = '/content/ExamCheatingDataset.zip'    # ← your zip filename
# with zipfile.ZipFile(ZIP_PATH, 'r') as z:
#     z.extractall('/content/raw')
# for root, dirs, _ in os.walk('/content/raw'):
#     if 'train' in dirs:
#         DATASET_ROOT = root
#         break
# ─────────────────────────────────────────────────────────────────

DATASET_ROOT = Path(DATASET_ROOT)
assert DATASET_ROOT.exists(), f'Folder not found: {DATASET_ROOT}'
assert (DATASET_ROOT / 'train').exists(), f'No train/ folder inside {DATASET_ROOT}'

print(f'✅ Dataset found: {DATASET_ROOT}')
print()

# Show what's inside
for split in ['train', 'valid', 'val', 'test']:
    p = DATASET_ROOT / split
    if not p.exists():
        continue
    classes = sorted([d for d in p.iterdir() if d.is_dir()])
    total   = sum(len(list(c.glob('*'))) for c in classes)
    print(f'  {split}/   {len(classes)} classes   {total} images')
    for c in classes:
        n = len(list(c.glob('*')))
        print(f'       • {c.name}: {n} images')

## 3 — Rename class folders to clean names

Creates a copy with spaces removed from folder names (YOLOv8 handles spaces,
but clean names make the live system display nicer labels).

In [ ]:
import shutil
from pathlib import Path

CLASS_MAP = {
    'normal act':     'NORMAL',
    'looking friend': 'LOOKING_AT_FRIEND',
    'giving object':  'GIVING_OBJECT',
    'giving code':    'GIVING_CODE',
    'cheating':       'CHEATING',
}

# Output folder — works on both Windows and Linux/Colab
import tempfile
CLEAN = Path(tempfile.gettempdir()) / 'ExamAnomaly'
if CLEAN.exists():
    shutil.rmtree(CLEAN)   # clear any previous run
CLEAN.mkdir(parents=True)

for split in ['train', 'valid', 'val', 'test']:
    src_split = DATASET_ROOT / split
    if not src_split.exists():
        continue
    for orig, clean in CLASS_MAP.items():
        src_cls = src_split / orig
        if not src_cls.exists():
            print(f'  ⚠️  Not found: {split}/{orig}  (skipping)')
            continue
        dst_cls = CLEAN / split / clean
        shutil.copytree(src_cls, dst_cls, dirs_exist_ok=True)
        n = len(list(dst_cls.glob('*')))
        print(f'  ✅ {split}/{orig}  →  {split}/{clean}  ({n} images)')

print(f'\n✅ Clean dataset at: {CLEAN}')

## 4 — Train

| Model | Speed | Accuracy |
|-------|-------|----------|
| `yolov8n-cls.pt` | fastest | lower |
| `yolov8s-cls.pt` | fast | **good ← start here** |
| `yolov8m-cls.pt` | medium | better |

Uses **CPU** when no GPU is available (slower, but works).

In [ ]:
import torch
from ultralytics import YOLO

# ── Configuration ───────────────────────────────────────────────
MODEL  = 'yolov8s-cls.pt'
EPOCHS = 80
BATCH  = 32
IMGSZ  = 224
# ───────────────────────────────────────────────────────────────

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print(f'Device: {"GPU: " + torch.cuda.get_device_name(0) if DEVICE == 0 else "CPU (no GPU found)"}')
if DEVICE == 'cpu':
    print('⚠️  Training on CPU is much slower. Consider using Google Colab with T4 GPU.')
    BATCH = 16   # smaller batch for CPU

# Output folder next to this notebook
import os
OUT_DIR = os.path.join(os.getcwd(), 'examguard_output')

model = YOLO(MODEL)
model.train(
    data    = str(CLEAN),
    epochs  = EPOCHS,
    batch   = BATCH,
    imgsz   = IMGSZ,
    device  = DEVICE,
    project = OUT_DIR,
    name    = 'cls_v1',
    patience= 15,
    save    = True,
    plots   = True,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.3,
    degrees=5, translate=0.1, scale=0.4, fliplr=0.5,
)

BEST = os.path.join(OUT_DIR, 'cls_v1', 'weights', 'best.pt')
print(f'\n✅ Done!  Best model: {BEST}')

## 5 — Evaluate

In [ ]:
from ultralytics import YOLO

model   = YOLO(BEST)
metrics = model.val(data=str(CLEAN), imgsz=IMGSZ)

acc = metrics.top1
print(f'\n📊 Top-1 Accuracy: {acc:.1%}')
print()
if   acc >= 0.90: print('🟢 Excellent — ready to deploy')
elif acc >= 0.80: print('🟡 Good — acceptable for production')
elif acc >= 0.70: print('🟠 Fair — try yolov8m-cls.pt or more epochs')
else:             print('🔴 Needs improvement — check class balance')

print('\nClass names:', list(model.names.values()))

## 6 — Test on sample images

In [ ]:
import random
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from ultralytics import YOLO

model = YOLO(BEST)

test_split = CLEAN / 'test'
if not test_split.exists():
    test_split = CLEAN / 'valid'

samples = []
for cls_dir in sorted(test_split.iterdir()):
    if not cls_dir.is_dir():
        continue
    imgs = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png'))
    if imgs:
        samples.append((random.choice(imgs), cls_dir.name))

n    = len(samples)
cols = min(n, 3)
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
axes = np.array(axes).flatten()

for i, (img_path, true_cls) in enumerate(samples):
    ax     = axes[i]
    result = model.predict(str(img_path), verbose=False)[0]
    pred   = model.names[result.probs.top1]
    conf   = float(result.probs.top1conf)
    ok     = pred == true_cls
    ax.imshow(np.array(Image.open(img_path).convert('RGB')))
    ax.axis('off')
    ax.set_title(
        f"{'✅' if ok else '❌'} True: {true_cls}\nPred: {pred} ({conf:.0%})",
        color='green' if ok else 'red', fontsize=9, fontweight='bold'
    )

for ax in axes[n:]:
    ax.axis('off')
plt.tight_layout()
plt.savefig('predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: predictions.png')

## 7 — View training curves

In [ ]:
import os
from IPython.display import Image as IPImage

plot_dir = os.path.join(OUT_DIR, 'cls_v1')
for fname in ['results.png', 'confusion_matrix_normalized.png']:
    p = os.path.join(plot_dir, fname)
    if os.path.exists(p):
        print(fname)
        display(IPImage(p, width=900))

## 8 — Copy model to your project

This copies `best.pt` directly into your ExamGuard project's `ml/weights/` folder.

In [ ]:
import shutil, os
from pathlib import Path

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Path to your project's ml/weights/ folder
PROJECT_WEIGHTS = r'D:\AI\GitHub\exam-anomaly-detection-system\ml\weights'
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

dest = Path(PROJECT_WEIGHTS) / 'exam_anomaly_classifier.pt'
shutil.copy(BEST, dest)

size_mb = dest.stat().st_size / 1e6
print(f'✅ Copied to: {dest}')
print(f'   Size: {size_mb:.1f} MB')
print()
print('Next steps:')
print('  1. Add to .env:  ML_MODEL_PATH=/ml/weights/exam_anomaly_classifier.pt')
print('  2. Rebuild:      docker-compose down && docker-compose up --build -d')

# ── If on Google Colab: download instead ──────────────────────────
# from google.colab import files
# shutil.copy(BEST, 'exam_anomaly_classifier.pt')
# files.download('exam_anomaly_classifier.pt')